# Hugging Face Audio Course — Unit 3: Transformer architectures for audio

Unit 3 is conceptual in the course. This notebook makes each idea **runnable** on real models:

| # | Concept | Model |
|---|---------|-------|
| 1 | Input representations: waveform vs log-mel | librosa |
| 2 | **CTC** decoding mechanics | `facebook/wav2vec2-base-960h` |
| 3 | **Seq2seq** encoder/decoder, task tokens, translation | `openai/whisper-small` |
| 4 | **Classification** with the Audio Spectrogram Transformer | `MIT/ast-finetuned-audioset-...` |

> **First run downloads ~1.3 GB** (whisper-small + AST) to the Hugging Face cache. wav2vec2-base
> and MINDS-14 are already cached from earlier units. CPU only; Whisper generation is a little slow.

In [ ]:
%matplotlib inline
import numpy as np
import torch
import librosa, librosa.display
import matplotlib.pyplot as plt
import IPython.display as ipd
from datasets import load_dataset, Audio

def load_minds14(name="en-AU", split="train"):
    kwargs = dict(path="PolyAI/minds14", name=name, split=split)
    try:
        return load_dataset(**kwargs)
    except Exception:
        return load_dataset(**kwargs, trust_remote_code=True)

# one English 16 kHz clip shared by sections 1-3
ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
array, sr = ds[0]["audio"]["array"], ds[0]["audio"]["sampling_rate"]
print("clip:", len(array), "samples @", sr, "Hz")
ipd.Audio(array, rate=sr)

## 1. Two ways audio enters a transformer
**Wav2Vec2** ingests the **raw waveform**; **Whisper** ingests an **80-bin log-mel spectrogram**.
The spectrogram is a far shorter sequence, which is why it is cheaper for the encoder.

In [ ]:
mel = librosa.feature.melspectrogram(y=array, sr=sr, n_mels=80, n_fft=400, hop_length=160, fmax=8000)
log_mel = librosa.power_to_db(mel, ref=np.max)
print("waveform:", len(array), "samples  ->  log-mel:", log_mel.shape,
      f"(~{len(array)//log_mel.shape[1]}x shorter)")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))
t = np.arange(len(array)) / sr
ax1.plot(t, array, lw=0.4); ax1.set(title="(A) Raw waveform (Wav2Vec2 input)", xlabel="time (s)")
img = librosa.display.specshow(log_mel, x_axis="time", y_axis="mel", sr=sr, hop_length=160, fmax=8000, ax=ax2)
ax2.set(title=f"(B) Log-mel 80 x {log_mel.shape[1]} (Whisper input)")
fig.colorbar(img, ax=ax2, format="%+2.0f dB"); fig.tight_layout(); plt.show()

## 2. CTC mechanics (Wav2Vec2)
A CTC model predicts one token per ~20 ms frame, then **collapses** the sequence: merge repeats and
drop the blank token (`<pad>`). Watch the raw per-frame string turn into text.

In [ ]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
proc = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h").eval()

inputs = proc(array, sampling_rate=sr, return_tensors="pt")
with torch.no_grad():
    logits = model(inputs.input_values).logits
frames = logits.shape[1]
print("logits:", tuple(logits.shape), "->", frames, "frames,", f"{frames/(len(array)/sr):.1f} frames/sec")

pred_ids = logits.argmax(-1)[0]
blank_id = proc.tokenizer.pad_token_id
delim = proc.tokenizer.word_delimiter_token
tokens = proc.tokenizer.convert_ids_to_tokens(pred_ids.tolist())
print("RAW per-frame tokens (note the <pad> blanks and repeats):")
print("".join(tokens))

def collapse(ids, blank):
    out, prev = [], None
    for i in ids:
        if i != prev and i != blank:
            out.append(i)
        prev = i
    return out

kept = proc.tokenizer.convert_ids_to_tokens(collapse(pred_ids.tolist(), blank_id))
manual = "".join(" " if t == delim else t for t in kept)
final = proc.batch_decode(pred_ids.unsqueeze(0))[0]
print("\ncollapsed   :", repr(manual.strip()))
print("batch_decode:", repr(final.strip()))
print("match:", manual.strip() == final.strip())

In [ ]:
plt.figure(figsize=(12, 3))
plt.step(range(frames), pred_ids.tolist(), where="mid", lw=0.8)
plt.axhline(blank_id, color="tab:red", ls="--", lw=0.8, label=f"blank id={blank_id}")
plt.xlabel("frame (~20 ms)"); plt.ylabel("argmax token id")
plt.title("CTC: predicted token per frame (spikes between blank runs)"); plt.legend(); plt.show()

## 3. Seq2seq mechanics (Whisper)
The encoder turns the log-mel into hidden states; the decoder generates tokens **autoregressively**,
primed with **task tokens** that say which language and task (transcribe / translate).

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
wproc = WhisperProcessor.from_pretrained("openai/whisper-small")
wmodel = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").eval()

feats = wproc(array, sampling_rate=sr, return_tensors="pt").input_features
print("input_features:", tuple(feats.shape), "(1, 80, 3000 = fixed 30s)")
with torch.no_grad():
    enc = wmodel.model.encoder(feats).last_hidden_state
print("encoder hidden states:", tuple(enc.shape))

prompt = wproc.get_decoder_prompt_ids(language="english", task="transcribe")
ids = [wproc.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")] + [i for _, i in prompt]
print("decoder task prompt:", wproc.tokenizer.convert_ids_to_tokens(ids))

with torch.no_grad():
    out = wmodel.generate(feats, language="english", task="transcribe", max_new_tokens=128)
print("transcription:", repr(wproc.decode(out[0], skip_special_tokens=True)))
print("Whisper vocab:", wmodel.config.vocab_size, "BPE tokens (vs CTC's 32 characters)")

In [ ]:
plt.figure(figsize=(12, 3))
plt.imshow(feats[0].numpy(), aspect="auto", origin="lower", cmap="magma")
plt.xlabel("frame"); plt.ylabel("mel bin"); plt.title("Whisper encoder input: log-mel (80 x 3000)")
plt.colorbar(); plt.show()

### 3d. Translation: German speech → English text
The same model can **translate** by switching the task token from `transcribe` to `translate`.

In [ ]:
de = load_minds14(name="de-DE").cast_column("audio", Audio(sampling_rate=16_000))
ex = de[0]
dfeats = wproc(ex["audio"]["array"], sampling_rate=16_000, return_tensors="pt").input_features
with torch.no_grad():
    transc = wmodel.generate(dfeats, language="german", task="transcribe", max_new_tokens=128)
    transl = wmodel.generate(dfeats, language="german", task="translate", max_new_tokens=128)
print("reference (de) :", repr(ex["transcription"]))
print("transcribe (de):", repr(wproc.decode(transc[0], skip_special_tokens=True)))
print("translate  (en):", repr(wproc.decode(transl[0], skip_special_tokens=True)))
print("gold english   :", repr(ex["english_transcription"]))
ipd.Audio(ex["audio"]["array"], rate=16_000)

## 4. Classification: the Audio Spectrogram Transformer
AST treats the spectrogram like an image: it splits it into 16×16 patches and runs a Vision
Transformer over them, then classifies into AudioSet's 527 sound classes.

In [ ]:
from transformers import ASTFeatureExtractor, ASTForAudioClassification
clip, sr0 = librosa.load(librosa.ex("trumpet"))
clip = librosa.resample(clip, orig_sr=sr0, target_sr=16_000)   # AST needs 16 kHz

ast_id = "MIT/ast-finetuned-audioset-10-10-0.4593"
fe = ASTFeatureExtractor.from_pretrained(ast_id)
ast = ASTForAudioClassification.from_pretrained(ast_id).eval()
ast_in = fe(clip, sampling_rate=16_000, return_tensors="pt")
print("AST input:", tuple(ast_in.input_values.shape), "(1, 1024 frames, 128 mel) -> 16x16 patches")
with torch.no_grad():
    probs = ast(**ast_in).logits.softmax(-1)[0]
top = probs.topk(5)
labels = [ast.config.id2label[i] for i in top.indices.tolist()]
for p, l in zip(top.values.tolist(), labels):
    print(f"  {p:.3f}  {l}")

plt.figure(figsize=(10, 4))
plt.barh(labels[::-1], top.values.tolist()[::-1], color="tab:purple")
plt.xlim(0, 1); plt.xlabel("probability"); plt.title("AST top-5 (trumpet clip)"); plt.show()
ipd.Audio(clip, rate=16_000)

In [ ]:
# Connect back to Unit 2 (config only, no 1.2 GB download):
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained("anton-l/xtreme_s_xlsr_300m_minds14")
print("Unit 2's intent classifier is a", cfg.architectures)
print("= the same Wav2Vec2 encoder, mean-pooled + a linear head (vs AST's spectrogram patches).")

---
### The three architectures at a glance
| | Input | Vocabulary / classes | Output style |
|---|---|---|---|
| **CTC** (Wav2Vec2) | raw waveform | 32 characters | per-frame predictions, collapsed |
| **Seq2seq** (Whisper) | log-mel | ~50k BPE tokens | autoregressive, cased + punctuated |
| **Classification** (AST) | spectrogram patches | 527 AudioSet classes | one pooled label set |

🎉 That's Unit 3. Next: Unit 4, build a music genre classifier.